# Task 3: Real-Time Echocardiogram Video Analysis

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

VIDEO_PATH = 'data/echo_sample.mp4'
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SHOW_LIVE = True
MAX_SCREENSHOTS = 10


## Enhancement pipeline for a single frame

In [ ]:
def gray_world_balance(bgr_image):
    b, g, r = cv2.split(bgr_image.astype(np.float64))
    gray_mean = (b.mean() + g.mean() + r.mean()) / 3.0
    b = np.clip(b * (gray_mean / (b.mean() + 1e-6)), 0, 255)
    g = np.clip(g * (gray_mean / (g.mean() + 1e-6)), 0, 255)
    r = np.clip(r * (gray_mean / (r.mean() + 1e-6)), 0, 255)
    return cv2.merge([b, g, r]).astype(np.uint8)


def log_transform(image):
    image_f = image.astype(np.float64)
    c = 255.0 / np.log(1 + image_f.max()) if image_f.max() > 0 else 1.0
    return np.clip(c * np.log(1 + image_f), 0, 255).astype(np.uint8)


def gamma_transform(image, gamma=0.7):
    normalized = image.astype(np.float64) / 255.0
    return np.clip(255.0 * np.power(normalized, gamma), 0, 255).astype(np.uint8)


def enhance_frame(gray_frame):
    equalized = cv2.equalizeHist(gray_frame)
    colored = cv2.applyColorMap(equalized, cv2.COLORMAP_JET)
    balanced = gray_world_balance(colored)
    logged = log_transform(balanced)
    return gamma_transform(logged, gamma=0.7)


## Real-time video loop (press 'q' to quit)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f'Could not open {VIDEO_PATH}'

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'Frames: {frame_count} | FPS: {fps:.1f}')

screenshot_interval = max(1, frame_count // MAX_SCREENSHOTS)
frame_idx = 0
saved_screenshots = 0
live_window_failed = False

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    enhanced = enhance_frame(gray)
    raw_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    combined = np.hstack([raw_bgr, enhanced])

    if SHOW_LIVE and not live_window_failed:
        try:
            cv2.imshow('Raw (left) vs Enhanced (right)', combined)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        except cv2.error:
            live_window_failed = True
            print('No display available; continuing headlessly (frames still saved to output/).')

    if frame_idx % screenshot_interval == 0 and saved_screenshots < MAX_SCREENSHOTS:
        cv2.imwrite(f'{OUTPUT_DIR}/frame_{frame_idx:04d}.png', combined)
        saved_screenshots += 1

    frame_idx += 1

cap.release()
try:
    cv2.destroyAllWindows()
except cv2.error:
    pass

print(f'Processed {frame_idx} frames, saved {saved_screenshots} screenshots to {OUTPUT_DIR}/')

saved = sorted(glob.glob(f'{OUTPUT_DIR}/frame_*.png'))
assert saved, 'No screenshots were saved — check that the video loaded correctly.'

preview = cv2.imread(saved[len(saved) // 2])
preview_path = os.path.join(OUTPUT_DIR, 'preview_frame.png')
cv2.imwrite(preview_path, preview)

plt.figure(figsize=(10, 5))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.title(f'Raw (left) vs Enhanced (right) — {os.path.basename(saved[len(saved) // 2])}')
plt.axis('off')
plt.show()


NameError: name 'VIDEO_PATH' is not defined